# Tests: `fasterai.sparse.sparsify_callback` (source `nbs/sparse/sparsify_callback.ipynb`)

In [ ]:
from fastcore.test import *
import torch
import torch.nn as nn
from fastai.learner import Learner
from fasterai.core.criteria import large_final
from fasterai.core.schedule import lin, one_shot
from fasterai.sparse.sparsify_callback import *

In [ ]:
from fastcore.test import *
import warnings

# Construction stores the fraction it was given
cb = SparsifyCallback(
    sparsity=0.5, granularity='weight', context='local',
    criteria=large_final, schedule=one_shot
)
test_eq(cb.sparsity, 0.5)
test_eq(cb.granularity, 'weight')
test_eq(cb.context, 'local')
test_eq(cb.current_sparsity, 0.0)

# A percent is read as x/100 for one release, and warns once at construction
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    cb_pct = SparsifyCallback(sparsity=50, granularity='weight', context='local',
                              criteria=large_final, schedule=one_shot)
test_eq(cb_pct.sparsity, 0.5)
test_eq(len(w), 1)
test_eq(w[0].category, FutureWarning)

# Dict-based sparsity, converted per layer
cb_dict = SparsifyCallback(
    sparsity={'conv1': 0.3, 'conv2': 0.6}, granularity='weight',
    context='local', criteria=large_final, schedule=lin
)
test_eq(cb_dict.sparsity, {'conv1': 0.3, 'conv2': 0.6})

# Out-of-range and non-numeric targets are refused, naming the layer
with ExceptionExpected(ValueError, regex="'conv2'"):
    SparsifyCallback(sparsity={'conv1': 0.3, 'conv2': 150}, granularity='weight',
                     context='local', criteria=large_final, schedule=lin)
with ExceptionExpected(TypeError):
    SparsifyCallback(sparsity='0.5', granularity='weight', context='local',
                     criteria=large_final, schedule=one_shot)

# _sparsity_value helper with float
cb.current_sparsity = 0.42
test_eq(cb._sparsity_value(), 0.42)

# _sparsity_value helper with dict
cb_dict.current_sparsity = {'a': 0.1, 'b': 0.2}
test_eq(cb_dict._sparsity_value(), 0.1)

# Reports are displayed as percentages
test_eq(cb._as_pct(0.5), '50.00%')
test_eq(cb._as_pct({'a': 0.1}), "{'a': '10.00%'}")

In [ ]:
#| slow
# Full training loop with SparsifyCallback — verify the fraction is reached, without warning
import os, tempfile, warnings
from torch.utils.data import TensorDataset
from fastai.data.core import DataLoaders

def _make_dls():
    _X = torch.randn(64, 3, 8, 8)
    _y = torch.randint(0, 10, (64,))
    return DataLoaders.from_dsets(
        TensorDataset(_X[:48], _y[:48]),
        TensorDataset(_X[48:], _y[48:]),
        bs=16, device='cpu'
    )

_model = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(16, 10)
)
_cb = SparsifyCallback(sparsity=0.5, granularity='weight', context='local',
                       criteria=large_final, schedule=one_shot)
_learn = Learner(_make_dls(), _model, loss_func=nn.CrossEntropyLoss(), cbs=[_cb])
with warnings.catch_warnings(record=True) as _w:
    warnings.simplefilter("always")
    _learn.fit(3)

for m in _model.modules():
    if isinstance(m, nn.Conv2d):
        test_close((m.weight == 0).float().mean().item() * 100, 50.0, eps=10.0)

# The scheduled intermediates are fractions too — nothing looks like a percent during training
_percent_warnings = [x for x in _w if 'looks like a percent' in str(x.message)]
test_eq(_percent_warnings, [])

# save_tickets writes a ticket per pruning step (regression: `copy` was missing at import time)
_model2 = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(16, 10)
)
_cb2 = SparsifyCallback(sparsity=0.5, granularity='weight', context='local',
                        criteria=large_final, schedule=one_shot, lth=True, save_tickets=True)
_learn2 = Learner(_make_dls(), _model2, loss_func=nn.CrossEntropyLoss(), cbs=[_cb2])
_cwd = os.getcwd()
with tempfile.TemporaryDirectory() as _d:
    os.chdir(_d)
    try:
        _learn2.fit(3)
        _tickets = [f for f in os.listdir(_d) if f.startswith('winning_ticket_')]
    finally:
        os.chdir(_cwd)
assert _tickets, 'no winning ticket was saved'
assert 'winning_ticket_50.00.pth' in _tickets, _tickets